In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

<font size="5" color="black">ch9. RAG</font>

# 0. 패키지 설치

In [3]:
# text → chunk pkg
%pip install -q langchain-chroma

Note: you may need to restart the kernel to use updated packages.


In [5]:
# langchain
%pip install -q langchain

Note: you may need to restart the kernel to use updated packages.


In [6]:
# text → chunk pkg
%pip install -q langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


# 1. 문서 로드

In [7]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./tax/소득세법(법률)(제21221호)(20260701).docx')
document = loader.load()
len(document)

CPU times: total: 6.09 s
Wall time: 6.49 s


1

# 2. 분할하며 로드


## 2.1 TokenTextSplitter
- 1500 token 단위로 겹쳐서 분할
```
 1. "o200k_base" :
    - 주요 모델: gpt-4o, gpt-4o-mini 등 최신 세대 모델
    - 설명: 최신 모델에 최적화된 인코딩 방식

 2. "cl100k_base" :
    - 주요 모델: gpt-4, gpt-3.5-turbo, text-embedding-3 등
    - 설명: ChatGPT 및 임베딩 모델에서 가장 널리 쓰이는 표준 인코딩

 3. "p50k_base" :
    - 주요 모델: text-davinci-003, Codex 등
    - 설명: 구버전 다빈치 모델 및 코드 처리에 최적화된 인코딩

 4. "r50k_base" (또는 "gpt2") :
    - 주요 모델: GPT-2, 구버전 GPT-3 모델
    - 설명: 가장 오래된 레거시 호환용 인코딩
```

In [10]:
%%time
from langchain_text_splitters import TokenTextSplitter
text_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=1500,
    chunk_overlap=200
)

documents = loader.load_and_split(text_splitter=text_splitter)

CPU times: total: 7.64 s
Wall time: 9.34 s


In [20]:
print(documents[0].page_content[1200:])

자로, 그 밖의 경우에는 1비거주자로 보아 이 법을 적용한다. 다만, 다음 각 호의 어느 하나에 해당하는 경우에는 소득구분에 따라 해당 단체의 각 구성원별로 이 법 또는 「법인세법」에 따라 소득에 대한 소득세 또는 법인세[해당 구성원이 「법인세법」에 따른 법인(법인으로 보는 단체를 포함한다)인 경우로 한정한다. 이하 이 조에서 같다]를 납부할 의무를 진다.<개정 2010. 12. 27., 2013. 1. 1., 2018. 12. 31.>

1. 구성원 간 이익의 분배비율이 정하여져 있고 해당 구성원별로 이익의 분배비율이 확인되는 경우

2. 구성원 간 이익의 분배비율이 정하여져 있지 아니하나 사실상 구성원별로 이익이 분배되는 것으로 확인되는 경우

④ 제3항에도 불구하고 해당 단체의 전체 구성원 중 일부 구성원의 분배비율만 확인되거나 일부 구성원에게만 이익이 분배되는 것으로 확인되는 경우에는 다음 각 호의 구분에 따라 소득세 또는 법인세를 납부할 의무를 진다.<신설 2018. 12. 31.>

1. 확인되는 부분:


In [21]:
# 오버랩 확인 (ex: 이익의 분배비율이 확인되는 경우)
print(documents[1].page_content[:200])

 이익의 분배비율이 확인되는 경우

2. 구성원 간 이익의 분배비율이 정하여져 있지 아니하나 사실상 구성원별로 이익이 분배되는 것으로 확인되는 경우

④ 제3항에도 불구하고 해당 단체의 전체 구성원 중 일부 구성원의 분배비율만 확인되거나 일부 구성원에게만 이익이 분배되는 것으로 확인되는 경우에는 다음 각 호의 구분에 따라 소득세 또는 법인세를 납부할 의무를


In [30]:
# 각 chunk의 글자 수
import numpy as np
print(np.array([len(document.page_content) for document in documents]).mean())
print(min([len(document.page_content) for document in documents]))
print(max([len(document.page_content) for document in documents]))

1611.5555555555557
1280
2048


## 2.2 RecursiveCharacterTextSplitter
- 1500자 단위로 겹쳐서 분할

In [31]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [32]:
loader = Docx2txtLoader('./tax/소득세법(법률)(제21221호)(20260701).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    separators=["\n\n\n\n", "\n\n", "\n", " "] # 분할 기준 우선순위
)

documents = loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print(runtime)

494.5946116447449


In [61]:
# 오버랩 확인 (6th chunk)
print(documents[6].page_content[1210:])

라. 「근로기준법」 또는 「선원법」에 따라 근로자ㆍ선원 및 그 유족이 받는 요양보상금, 휴업보상금, 상병보상금(傷病補償金), 일시보상금, 장해보상금, 유족보상금, 행방불명보상금, 소지품 유실보상금, 장의비 및 장제비


In [64]:
# 오버랩 확인 (7th chunk)
print(documents[7].page_content[:120])

라. 「근로기준법」 또는 「선원법」에 따라 근로자ㆍ선원 및 그 유족이 받는 요양보상금, 휴업보상금, 상병보상금(傷病補償金), 일시보상금, 장해보상금, 유족보상금, 행방불명보상금, 소지품 유실보상금, 장의비 및 장제비


In [65]:
# 각 chunk의 글자 수
import numpy as np
print(np.array([len(document.page_content) for document in documents]).mean())
print(min([len(document.page_content) for document in documents]))
print(max([len(document.page_content) for document in documents]))

1118.2435897435898
40
1495


# 3. 분할된 문서 임베딩 → vector Store(DB)에 저장

In [70]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

import os
load_dotenv()

embedding = OpenAIEmbeddings(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='text-embedding-3-large'
)

In [71]:
len(embedding.embed_query("소득세법의 예시 구문"))

3072

In [76]:
vectors = embedding.embed_documents([document.page_content for document in documents])

In [77]:
len(vectors[0]), len(vectors[1])

(3072, 3072)

In [78]:
%%time
from langchain_chroma import Chroma

# 데이터 최초 저장 시
database = Chroma.from_documents(
    documents=documents, # 분할된 documents
    embedding=embedding, # 임베딩 객체
    collection_name='tax_collection', # 생략 시 랜덤 생성
    persist_directory='./chroma' # 생략 시 local file 저장 불가 (인메모리 변수 저장 처리)
)

CPU times: total: 3.3 s
Wall time: 6.13 s


In [79]:
%%time

# 데이터 로드 시
database = Chroma(
    embedding_function=embedding,
    collection_name='tax_collection',
    persist_directory='./chroma'
)

CPU times: total: 31.2 ms
Wall time: 33.3 ms


In [88]:
results = database.get(include=['metadatas', 'embeddings', 'documents'])
print(f'데이터 수 : {len(results["ids"])}')
print(f'문서 임베딩 차원 수 : {len(results["embeddings"][0])}')

데이터 수 : 234
문서 임베딩 차원 수 : 3072


# 4. Retrieval 작업 (유사도 검색)

In [96]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(
    query=query, 
    k=2 # 기본값 : 4
)

In [97]:
len(retrieved_docs)

2

In [98]:
retrieved_doc = "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

# 5. 질문과 doc을 LLM에 전달 및 답변 생성

In [99]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano')

In [100]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가 입니다
- 다음의 [context]를 참조하여 사용자의 질문에 답변하세요
[context]
{retrieved_doc}
질문 : {query}"""

ai_msg = llm.invoke(prompt)

In [103]:
print(ai_msg.usage_metadata)

{'input_tokens': 1399, 'output_tokens': 658, 'total_tokens': 2057, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [110]:
from langchain_core.prompts import ChatPromptTemplate

promptTemplate = ChatPromptTemplate([
    ('system','당신은 최고의 한국 소득세법 전문가입니다'),
    ('human',f'''다음 문맥을 참고하여 5문장 이내로 축약하여 질문에 답변하세요. 답을 모르면 모른다고 답변하세요.
    질문 : {{question}}, 문맥 : {{context}}, 답변 : ''')
])

In [111]:
prompt = promptTemplate.invoke(
    {
    'context':retrieved_doc,
    'question':query
    }
)

In [112]:
ai_message = llm.invoke(prompt)

In [114]:
# StrOutputParser 활용
from langchain_core.output_parsers import StrOutputParser
outputParser = StrOutputParser()
outputParser.invoke(ai_message)

'연봉 5천만원인 직장인의 소득세는 여러 공제와 세율을 고려해야 하며, 일반적으로 근로소득세율은 소득구간에 따라 차등적용됩니다. 구체적인 세액 계산을 위해서는 인적공제, 근로소득공제, 표준공제 등을 반영해야 하며, 세율은 누진세 구조입니다. 따라서 정확한 세액을 계산하려면 구체적인 공제 내역이 필요합니다. 참고로, 본문에서는 근로소득에 대한 과세 표준이나 세율에 관한 구체적인 내용이 명시되어 있지 않으므로, 정확한 소득세 금액 산출은 어렵습니다.'

In [115]:
# 한번에
outputParser.invoke(llm.invoke(promptTemplate.invoke({
    'context':retrieved_doc,'question':query
    })))

'연봉 5천만원인 직장인의 소득세액은 정확한 계산을 위해 여러 공제와 세율을 고려해야 하지만, 일반적으로 근로소득세율표에 따라 대략 8~15% 내외가 예상됩니다. 다만, 일정 부분은 비과세 소득이나 공제, 세율 적용에 따라 차이가 발생할 수 있습니다. 제공된 문맥에서는 구체적인 세율이나 세액 계산 방법이 명시되어 있지 않으므로, 구체적인 금액 산출이 어렵습니다. 따라서 정확한 소득세금액은 전문가와 상담하거나 국세청 계산 공식에 따라 산출하는 것이 바람직합니다.'

In [116]:
# LCEL 활용
chain = promptTemplate | llm | outputParser
chain.invoke({'context':retrieved_doc,'question':query})

'연봉 5천만원인 직장인의 소득세는 정확한 세액 계산을 위해 과세표준, 근로소득 공제, 기본공제, 기초세액공제 등을 고려해야 합니다. 일반적인 경우 5천만원 연봉에서는 근로소득공제와 표준세율을 적용하여 계산하며, 이를 기초로 납부 세액이 도출됩니다. 하지만 개별 공제 항목이나 특별세액공제 여부에 따라 최종 세액은 달라질 수 있습니다. 따라서 구체적 계산을 위해 세무서 또는 세무전문가의 상담이 필요합니다.'

LangChain으로 전달

In [117]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

In [118]:
# 임베딩 & LLM 초기화
load_dotenv()
embedding = OpenAIEmbeddings(model = 'text-embedding-3-large')
llm = ChatOpenAI(model = 'gpt-4o-mini')

# vectorDB load
vectorStore = Chroma(
    embedding_function=embedding,
    persist_directory='./chroma',
    collection_name='tax-collection'
)

# Retriever 생성
# 1 : 유사도 상위 k개
retriever = vectorStore.as_retriever(
    search_type='similarity',
    search_kwargs={'k':3}
)
# 2 : 유사도 상위 fetch_k개 중 내용이 겹치지 않는 k개
# retriever = vectorStore.as_retriever(
#     search_type='mmr',
#     search_kwargs={
#         'k':3,
#         'fetch_k':10,
#         'lambda_mult':0.5
#     }    
# )
# 3 : score_threshold 이상의 유사도 상위 k개
# retriever = vectorStore.as_retriever(
#     search_type='similarity_score_threshold',
#     search_kwargs={
#         'k':3,
#         'score_threshold':0.4
#     }
# )

# 프롬프트 템플릿
template = f'''당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 최대 5문장으로 답하세요. 답을 모르면 모른다고 답하세요.
질문 : {{query}}
문맥 : {{context}}
답변 : '''
prompt = ChatPromptTemplate.from_template(template)

# 검색한 문서 텍스트 변환
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

In [119]:
# RAG chain
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    |prompt
    |llm
    |StrOutputParser()
)

rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼만가요?')

'소득세는 개인의 총소득에 따라 다르게 계산됩니다. 연봉 5천만원인 직장인의 경우, 기본 공제와 각종 세액 공제를 고려하면 실제 과세소득이 줄어들 수 있습니다. 2023년 기준으로, 소득세율은 6%에서 시작해 차등적으로 적용되므로 정확한 세액 계산을 위해서는 공제 항목과 세액을 정확히 반영해야 합니다. 간단한 계산으로, 기본적인 공제를 적용하면 대략적으로 연봉 5천만원의 소득세는 약 300만원에서 400만원 사이에 해당할 것으로 예상됩니다. 그러나 개인의 상황에 따라 차이가 있으니, 세무 전문가와의 상담을 추천합니다.'